# YOLO Receipt Text Detection - Kaggle Training

Train YOLOv8 detector for receipt text detection on Kaggle GPUs for faster training.

**Features:**
- Trains on combined Vietnamese + English receipt dataset
- Uses GPU acceleration for fast training
- Automatic model evaluation and visualization
- Saves best weights for inference

**Preparation:**
1. Upload your data to Kaggle notebook as input
2. Or provide paths to existing datasets
3. Run all cells sequentially

## 1. Import Required Libraries and Setup

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess
import json

# Check GPU availability
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("GPU Memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

# Set working directory to project root
PROJECT_ROOT = "/kaggle/input/ocr-receipt"  # Change this to your data path
if not os.path.exists(PROJECT_ROOT):
    PROJECT_ROOT = "/kaggle/working"
    print(f"Warning: Using working directory: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f"\nProject Root: {PROJECT_ROOT}")
print(f"Working Directory: {os.getcwd()}")

## 2. Configure Paths for Data and Artifacts

In [ ]:
# Define paths
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, "artifacts")
DETECTOR_DIR = os.path.join(ARTIFACTS_DIR, "detector_runs", "yolo_textdet_kaggle")
WEIGHTS_DIR = os.path.join(DETECTOR_DIR, "weights")

# Create necessary directories
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(DETECTOR_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Paths to datasets
COMBINED_YAML = os.path.join(DATA_DIR, "combined_receipt.yaml")
EN_RECEIPT_DIR = os.path.join(DATA_DIR, "en_receipt")
VN_RECEIPT_DIR = os.path.join(DATA_DIR, "vn_receipt")

print("📁 Configured Paths:")
print(f"  Data Directory: {DATA_DIR}")
print(f"  Artifacts Directory: {ARTIFACTS_DIR}")
print(f"  Detector Output: {DETECTOR_DIR}")
print(f"  Combined YAML: {COMBINED_YAML}")
print(f"\n✓ All directories created successfully!")

## 3. Verify Dataset Structure

In [ ]:
def verify_dataset_structure():
    """Verify that all required dataset files exist"""
    print("📊 Verifying Dataset Structure...\n")
    
    required_paths = {
        "English Receipt Train Images": os.path.join(EN_RECEIPT_DIR, "images", "train"),
        "English Receipt Valid Images": os.path.join(EN_RECEIPT_DIR, "images", "valid"),
        "English Receipt Train Labels": os.path.join(EN_RECEIPT_DIR, "labels", "train"),
        "English Receipt Valid Labels": os.path.join(EN_RECEIPT_DIR, "labels", "valid"),
        "Vietnamese Receipt Train Images": os.path.join(VN_RECEIPT_DIR, "images", "train"),
        "Vietnamese Receipt Val Images": os.path.join(VN_RECEIPT_DIR, "images", "val"),
        "Vietnamese Receipt Train Labels": os.path.join(VN_RECEIPT_DIR, "labels", "train"),
        "Vietnamese Receipt Val Labels": os.path.join(VN_RECEIPT_DIR, "labels", "val"),
        "Combined YAML Config": COMBINED_YAML,
    }
    
    all_exist = True
    for name, path in required_paths.items():
        exists = os.path.exists(path)
        status = "✓" if exists else "✗"
        print(f"  {status} {name}: {path}")
        if not exists:
            all_exist = False
    
    if all_exist:
        print("\n✓ All dataset files verified!")
    else:
        print("\n⚠ Warning: Some dataset files are missing. Please ensure all data is properly uploaded.")
        print("  See HOW_TO_RUN.md prerequisites for required structure.")
    
    return all_exist

# Verify dataset
dataset_ready = verify_dataset_structure()

# Count data samples if dataset exists
if dataset_ready:
    en_train_imgs = len(os.listdir(os.path.join(EN_RECEIPT_DIR, "images", "train")))
    en_valid_imgs = len(os.listdir(os.path.join(EN_RECEIPT_DIR, "images", "valid")))
    vn_train_imgs = len(os.listdir(os.path.join(VN_RECEIPT_DIR, "images", "train")))
    vn_val_imgs = len(os.listdir(os.path.join(VN_RECEIPT_DIR, "images", "val")))
    
    print(f"\n📈 Dataset Statistics:")
    print(f"  EN Receipt - Train: {en_train_imgs} images, Valid: {en_valid_imgs} images")
    print(f"  VN Receipt - Train: {vn_train_imgs} images, Val: {vn_val_imgs} images")
    print(f"  Total: {en_train_imgs + en_valid_imgs + vn_train_imgs + vn_val_imgs} images")

## 4. Install YOLOv8

In [ ]:
import subprocess
import sys

print("📦 Installing YOLOv8...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

print("✓ YOLOv8 installed successfully!")

# Verify installation
from ultralytics import YOLO
print(f"\n✓ YOLO imported successfully")

# Display available model sizes
print("\n📋 Available YOLOv8 Model Sizes:")
model_sizes = ["n", "s", "m", "l", "x"]
for size in model_sizes:
    print(f"  - yolov8{size}.pt (recommended for production)")
print("\n💡 Nano (n) is fastest, XL (x) is most accurate. Balanced choice: Small (s) or Medium (m)")

## 5. Load Combined YOLO Dataset Configuration

In [ ]:
import yaml

def load_dataset_config(yaml_path):
    """Load and validate YOLO dataset configuration"""
    try:
        with open(yaml_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
    except Exception as e:
        print(f"❌ Error loading YOLO config: {e}")
        return None

print("📋 Loading Dataset Configuration...\n")

if os.path.exists(COMBINED_YAML):
    config = load_dataset_config(COMBINED_YAML)
    
    if config:
        print("✓ Combined Receipt Configuration Loaded:")
        print(f"  Path: {COMBINED_YAML}")
        print(f"  Train path: {config.get('path', 'N/A')}")
        print(f"  NC (num classes): {config.get('nc', 'N/A')}")
        print(f"  Names: {config.get('names', 'N/A')}")
        
        # Validate paths in config
        print("\n🔍 Validating dataset paths...")
        path_keys = ['train', 'val', 'test']
        for key in path_keys:
            if key in config:
                full_path = os.path.join(PROJECT_ROOT, config[key])
                exists = os.path.exists(full_path)
                status = "✓" if exists else "✗"
                print(f"  {status} {key}: {config[key]}")
else:
    print(f"⚠ Warning: combined_receipt.yaml not found at {COMBINED_YAML}")
    print("  Run 'python main.py preprocess' first to generate the combined dataset.")

## 6. Train YOLO Detector

⚠️ **This cell will take significant time (30-120 minutes depending on GPU and settings)**

Training parameters can be adjusted below. Default settings are optimized for Kaggle GPUs.

In [ ]:
from ultralytics import YOLO
import torch

# ============= TRAINING CONFIGURATION =============
# Adjust these parameters based on your GPU and requirements
TRAINING_CONFIG = {
    'model': 'yolov8s.pt',        # Model size: n=nano, s=small, m=medium, l=large, x=xlarge
    'epochs': 50,                  # Number of training epochs
    'batch_size': 16,              # Batch size (adjust if GPU memory insufficient)
    'imgsz': 640,                  # Image size (640x640)
    'patience': 5,                 # Early stopping patience
    'device': 0,                   # GPU device (0 = first GPU)
    'lr0': 0.01,                   # Initial learning rate
    'lrf': 0.01,                   # Final learning rate ratio
    'momentum': 0.937,             # SGD momentum
    'weight_decay': 0.0005,        # Weight decay
    'warmup_epochs': 3.0,          # Warmup epochs
    'warmup_momentum': 0.8,        # Warmup momentum
    'warmup_bias_lr': 0.1,         # Warmup bias learning rate
}

print("🚀 YOLO Detector Training Configuration:")
print("=" * 50)
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")
print("=" * 50)

# Verify YAML exists before training
if not os.path.exists(COMBINED_YAML):
    print("\n❌ ERROR: combined_receipt.yaml not found!")
    print("Please run preprocessing step first.")
else:
    print("\n🎯 Starting YOLO Training...")
    print(f"Dataset: {COMBINED_YAML}")
    print(f"Output: {DETECTOR_DIR}")
    
    try:
        # Load YOLOv8 model
        model = YOLO(TRAINING_CONFIG['model'])
        
        # Train the model
        results = model.train(
            data=COMBINED_YAML,
            epochs=TRAINING_CONFIG['epochs'],
            imgsz=TRAINING_CONFIG['imgsz'],
            batch=TRAINING_CONFIG['batch_size'],
            device=TRAINING_CONFIG['device'],
            project=ARTIFACTS_DIR,
            name='yolo_textdet_kaggle',
            exist_ok=True,
            patience=TRAINING_CONFIG['patience'],
            save=True,
            save_period=10,           # Save checkpoint every 10 epochs
            verbose=True,
            lr0=TRAINING_CONFIG['lr0'],
            lrf=TRAINING_CONFIG['lrf'],
            momentum=TRAINING_CONFIG['momentum'],
            weight_decay=TRAINING_CONFIG['weight_decay'],
            warmup_epochs=TRAINING_CONFIG['warmup_epochs'],
            warmup_momentum=TRAINING_CONFIG['warmup_momentum'],
            warmup_bias_lr=TRAINING_CONFIG['warmup_bias_lr'],
        )
        
        print("\n✅ Training completed successfully!")
        print(f"Results saved to: {DETECTOR_DIR}")
        
    except Exception as e:
        print(f"\n❌ Training error: {e}")
        import traceback
        traceback.print_exc()

## 7. Evaluate Model Performance

In [ ]:
import os
import glob
from pathlib import Path

# Find the best model weights
best_weights = os.path.join(DETECTOR_DIR, "weights", "best.pt")
last_weights = os.path.join(DETECTOR_DIR, "weights", "last.pt")

print("🏆 Model Evaluation")
print("=" * 50)

if os.path.exists(best_weights):
    print(f"✓ Best weights found: {best_weights}")
    
    # Load best model
    best_model = YOLO(best_weights)
    
    # Run validation on validation set
    print("\n📊 Running validation on test set...")
    val_results = best_model.val(data=COMBINED_YAML)
    
    print("\n✅ Validation Results:")
    print(f"  mAP50: {val_results.box.map50:.4f}")
    print(f"  mAP50-95: {val_results.box.map:.4f}")
    print(f"  Precision: {val_results.box.mp:.4f}")
    print(f"  Recall: {val_results.box.mr:.4f}")
else:
    print("⚠ Best weights not found. Training may not have completed.")
    print(f"  Expected path: {best_weights}")

# Display training artifacts
print("\n📁 Training Artifacts:")
results_dir = DETECTOR_DIR
if os.path.exists(results_dir):
    artifacts = {
        'Training Curves': os.path.join(results_dir, 'results.png'),
        'Confusion Matrix': os.path.join(results_dir, 'confusion_matrix.png'),
        'Results CSV': os.path.join(results_dir, 'results.csv'),
        'Best Weights': best_weights,
        'Last Weights': last_weights,
    }
    
    for name, path in artifacts.items():
        if os.path.exists(path):
            print(f"  ✓ {name}: {path}")
        else:
            print(f"  ○ {name}: (not found)")
else:
    print(f"  Training directory not found: {results_dir}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Display training curves
results_png = os.path.join(DETECTOR_DIR, "results.png")
confusion_matrix_png = os.path.join(DETECTOR_DIR, "confusion_matrix.png")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

try:
    if os.path.exists(results_png):
        img = Image.open(results_png)
        axes[0].imshow(img)
        axes[0].set_title("Training Results")
        axes[0].axis('off')
    else:
        axes[0].text(0.5, 0.5, 'Results plot not found', ha='center', va='center')
        axes[0].axis('off')
except Exception as e:
    axes[0].text(0.5, 0.5, f'Error loading: {e}', ha='center', va='center')
    axes[0].axis('off')

try:
    if os.path.exists(confusion_matrix_png):
        img = Image.open(confusion_matrix_png)
        axes[1].imshow(img)
        axes[1].set_title("Confusion Matrix")
        axes[1].axis('off')
    else:
        axes[1].text(0.5, 0.5, 'Confusion matrix not found', ha='center', va='center')
        axes[1].axis('off')
except Exception as e:
    axes[1].text(0.5, 0.5, f'Error loading: {e}', ha='center', va='center')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(DETECTOR_DIR, 'evaluation_summary.png'), dpi=100, bbox_inches='tight')
plt.show()

print("✓ Evaluation visualization saved!")

## 8. Save and Export Best Weights

Download the trained weights and results to use for inference.

In [ ]:
import shutil
import json
from datetime import datetime

print("💾 Saving and Exporting Trained Models")
print("=" * 50)

# Create export directory
export_dir = os.path.join(ARTIFACTS_DIR, "yolo_export")
os.makedirs(export_dir, exist_ok=True)

# Copy best weights to export directory
if os.path.exists(best_weights):
    export_best = os.path.join(export_dir, "yolov8_best.pt")
    shutil.copy2(best_weights, export_best)
    print(f"✓ Best weights exported: {export_best}")
else:
    print("⚠ Best weights not found")

# Copy training summary
results_csv = os.path.join(DETECTOR_DIR, "results.csv")
if os.path.exists(results_csv):
    export_results = os.path.join(export_dir, "training_results.csv")
    shutil.copy2(results_csv, export_results)
    print(f"✓ Training results exported: {export_results}")

# Create training summary JSON
summary = {
    "training_date": datetime.now().isoformat(),
    "model_type": "YOLOv8",
    "model_size": TRAINING_CONFIG['model'],
    "epochs": TRAINING_CONFIG['epochs'],
    "batch_size": TRAINING_CONFIG['batch_size'],
    "image_size": TRAINING_CONFIG['imgsz'],
    "best_weights": export_best if os.path.exists(best_weights) else None,
    "output_directory": DETECTOR_DIR,
}

summary_file = os.path.join(export_dir, "training_config.json")
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Training configuration saved: {summary_file}")

print("\n" + "=" * 50)
print("📦 Export Summary:")
print(f"  Export Directory: {export_dir}")
print(f"  Best Weights: {export_best if os.path.exists(best_weights) else 'Not found'}")
print(f"  Training Results: {export_results if os.path.exists(results_csv) else 'Not found'}")
print(f"  Config: {summary_file}")

print("\n💡 Next Steps:")
print("  1. Download the exported weights from Kaggle output")
print("  2. Place yolov8_best.pt in your local artifacts/detector_runs/yolo_textdet/weights/")
print("  3. Use it for inference: python main.py infer --detector artifacts/detector_runs/yolo_textdet/weights/best.pt ...")

# List all files in export directory
print("\n📁 Files in export directory:")
for file in os.listdir(export_dir):
    file_path = os.path.join(export_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  - {file} ({size_mb:.2f} MB)")

## 9. Quick Reference and Tips

### Training Complete! 🎉

**Key Files Generated:**
- `artifacts/detector_runs/yolo_textdet_kaggle/weights/best.pt` - Best trained model
- `artifacts/yolo_export/yolov8_best.pt` - Exported weights (download this)
- `artifacts/detector_runs/yolo_textdet_kaggle/results.png` - Training curves
- `artifacts/detector_runs/yolo_textdet_kaggle/confusion_matrix.png` - Evaluation metrics

### How to Use Trained Weights Locally

After downloading `yolov8_best.pt` from Kaggle:

```bash
# Copy to your local project
cp yolov8_best.pt artifacts/detector_runs/yolo_textdet/weights/best.pt

# Run inference on a receipt image
python main.py infer \
  --image path/to/receipt.jpg \
  --detector artifacts/detector_runs/yolo_textdet/weights/best.pt \
  --recognizer artifacts/checkpoints/recognition_best.pt
```

### Troubleshooting

**Out of Memory?**
- Reduce batch_size (e.g., from 16 to 8)
- Reduce image size (e.g., from 640 to 416)
- Use smaller model (n instead of s)

**Training seems stuck?**
- Check GPU usage with `nvidia-smi` in a terminal cell
- Verify dataset paths are correct
- Ensure combined_receipt.yaml exists with correct paths

**Need to resume training?**
- Modify the model path to point to your last checkpoint
- YOLOv8 will automatically resume from epoch where it stopped

### Performance Tips

| Model Size | Speed | Accuracy | GPU Memory |
|-----------|-------|----------|-----------|
| Nano (n) | ⚡⚡⚡ | ★★☆ | ~1GB |
| Small (s) | ⚡⚡ | ★★★ | ~2GB |
| Medium (m) | ⚡ | ★★★★ | ~4GB |
| Large (l) | 🐢 | ★★★★★ | ~8GB |

**Recommended for Kaggle:** Small (s) or Medium (m) model size provides good balance.